In [297]:
import pandas as pd
import numpy as np
import os
import math
from datetime import datetime
from IPython.display import display, Markdown

pd.set_option('display.max_colwidth', None)

In [298]:
# Load mom and dad's bank statements
data_path = r'data/'
sabbadel_columns = ['date', 'name', 'fin_date', 'value', 'account', 'nan', 'code']
bank_statements = []

for file in os.listdir(data_path):
    if file.endswith(".txt"):
        filepath = os.path.join(data_path, file)
        bank_statement = pd.read_csv(filepath, 
                                     sep='|', 
                                     header=None, 
                                     names=sabbadel_columns, 
                                     encoding='latin1')
        if 'ma' in file:
            bank_statement['who'] = 'ma'
        if 'pa' in file:
            bank_statement['who'] = 'pa'

        bank_statements.append(bank_statement)

parent_statements = pd.concat(bank_statements)

# Correct values
parent_statements['date'] = pd.to_datetime(parent_statements['date'], dayfirst=True)
parent_statements = parent_statements.drop(['nan', 'fin_date', 'account', 'code'], axis=1)
parent_statements = parent_statements.drop_duplicates()


In [299]:
# Load my bank statements
bank_statements = []

for file in os.listdir(data_path):
    if file.endswith(".xls"):
        filepath = os.path.join(data_path, file)
        bank_statement = pd.read_excel(filepath, skiprows=7) # the first 7 rows are account statistics, not transactions
        bank_statements.append(bank_statement)

my_statements = pd.concat(bank_statements)

# Standardize my bank statements
my_statements['date'] = pd.to_datetime(my_statements['FECHA VALOR'], dayfirst=True)
my_statements = my_statements.drop(['FECHA OPERACIÓN', 'SALDO', 'FECHA VALOR'], axis=1)
my_statements = my_statements.rename(columns={
    'CONCEPTO': 'name',
    'IMPORTE EUR': 'value'
})
my_statements = my_statements.drop_duplicates()[['date','name','value']]

In [300]:
# Define exclusion and inclusion strings and function
include = [
    "SUPER VINI",
    "ALCAMPO",
    "MARKET",
    "FRUTA",
    "mercadona",
    "carniceria",
    "veterinaria",
    "carref",
    "hiper",
    "papa jonhs",
    "polleria",
    "organic shop",
    "LIDL",
    "alejandro",
    "SANTIAGO",
    "Amon",
    "SEGUROS ADESLAS",
    "ELECTRICIDAD IBERDROLA",
    "AGUA CANAL",
    "LAS ROZAS",
    "ELECTRICIDAD FACTOR ENERGIA",
    "PESCADERIA",
    "dulce",
    "quesos",
    "paladar",
    "IBER LIMOSTAR",
    "TELEFONOS SIMYO",
    "HELADERIA"
    ]
include = [s.lower().strip() for s in include]

exclude = [
    "IKEA",
    "EL CORTE INGLES",
    "LEROY MERLIN",
    "AUTOESCUELA",
    "XE EUROPE",
    "Decorabano-Armilla",
    "Arciniega",
    "TRASPASO",
    "IBERIA",
    "COMISIONES",
    "INTERESES",
    "RECARGA",
    "IBERDROLA OTROS 2",
    "Ibkr",
    ]
exclude = [s.lower().strip() for s in exclude]

def filter_rows(df):
    df_in  = df[df['name'].str.contains('|'.join(include), case=False, na=False)]
    df_in = df_in[~df_in['name'].str.contains('|'.join(exclude), case=False, na=False)]

    df_ex  = df[~df['name'].isin(df_in['name'])]

    return df_in, df_ex


parent_included, parent_excluded = filter_rows(parent_statements)
my_included, my_excluded = filter_rows(my_statements)

In [301]:
# Define calculation functions
def get_previous_month(month, year):
    month -= 1
    if month == 0:
        month = 12
        year = year - 1

    return month, year


def filter_statements_by_date(df, month, year):
    months_df = df[(df['date'].dt.month == month) & (df['date'].dt.year == year)]

    return months_df


def calculate_payment(month, year, remainder):
    my_costs = filter_statements_by_date(my_included, month, year)
    parent_costs = filter_statements_by_date(parent_included, month, year)

    cash_paid = pd.read_csv(r'data/cash_paid.csv')
    previous_payment = cash_paid[cash_paid['month'] == month]['cash']

    total_costs = parent_costs['value'].sum() - my_costs['value'].sum() - previous_payment + remainder
    current_payment = round(-total_costs/4, -1)

    return current_payment


month, year = get_previous_month(datetime.now().month, datetime.now().year)
new_payment = calculate_payment(month, year, remainder=3).item() ###################################################### UPDATE REMAINDER ##################################################
print(f'{new_payment} EUR for {datetime.now().strftime("%B")}')

# Update cash_paid
cash_paid = pd.read_csv(r'data/cash_paid.csv')
if cash_paid.tail(1)['month'].iloc[0] == month:
    pd.concat([cash_paid, pd.DataFrame({'cash': new_payment, 
                                        'month': datetime.now().month, 
                                        'year': datetime.now().year}
                                        , index=[0])]).to_csv(r'data/cash_paid.csv', index=False)
elif not cash_paid.empty and (cash_paid.tail(1)['month'].iloc[0] == month + 1) and (cash_paid.tail(1)['cash'].iloc[0] != new_payment):
    cash_paid.loc[cash_paid.index[-1], 'cash'] = new_payment 
    cash_paid.to_csv(r'data/cash_paid.csv', index=False)


330.0 EUR for August


In [302]:
pd.set_option('display.max_rows', None)

print()
display(Markdown("# Excluded from calculation"))
display(filter_statements_by_date(parent_excluded, month, year))

print()
display(Markdown("# Included in calculation"))
display(filter_statements_by_date(parent_included, month, year))

# Excluded from calculation

,date,name,value,who
2,2024-07-29,COMPRA TARJ. 5402XXXXXXXX9016 IKEA LAS ROZAS HFB-CARRETERA DE,-1.50,ma
3,2024-07-29,COMPRA TARJ. 5402XXXXXXXX9016 IKEA LAS ROZAS HFB-CARRETERA DE,-3.50,ma
4,2024-07-29,ABONO BIZUM DE JANETH VERONICA MONTALVAN,20.00,ma
6,2024-07-26,COMPRA TARJ. 5402XXXXXXXX9016 MUY MUCHO ATOCHA-MADRID,-3.49,ma
9,2024-07-23,COMPRA TARJ. 5402XXXXXXXX9016 PRIMOR GRAN VIA-MADRID,-33.81,ma
12,2024-07-17,COMPRA TARJ. 5402XXXXXXXX9016 GIOELIA MADRID-MADRID,-8.70,ma
13,2024-07-17,COMPRA TARJ. 5402XXXXXXXX9016 JULIAN LOPEZ MADRID-MADRID,-28.40,ma
15,2024-07-15,COMPRA TARJ. 5402XXXXXXXX9016 DECATHLON ESPANA SAU-MADRID,-5.99,ma
16,2024-07-15,COMPRA TARJ. 5402XXXXXXXX9016 GRANIER ATOCHA-MADRID,-11.40,ma
17,2024-07-15,COMPRA TARJ. 5402XXXXXXXX9016 MISS KITS MADRID-MADRID,-25.88,ma


# Included in calculation

,date,name,value,who
0,2024-07-31,COMPRA TARJ. 5402XXXXXXXX9016 MARKET LAS ROZA-LAS ROZAS,-14.57,ma
1,2024-07-31,COMPRA TARJ. 5402XXXXXXXX9016 MERCADONA C.C. EUROPOLIS-LAS ROZAS,-27.30,ma
5,2024-07-26,COMPRA TARJ. 5402XXXXXXXX9016 PAPA JOHNS - LAS ROZAS-LAS ROZAS DE,-19.98,ma
7,2024-07-23,COMPRA TARJ. 5402XXXXXXXX9016 FRUTAS AZAHARA-COLLADO VILLA,-3.39,ma
8,2024-07-23,COMPRA TARJ. 5402XXXXXXXX9016 HELADERIA DON NINO-MADRID,-47.00,ma
10,2024-07-22,COMPRA TARJ. 5402XXXXXXXX9016 CLINICA VETERINARIA ESCLA-LAS ROZAS DE,-38.00,ma
11,2024-07-17,COMPRA TARJ. 5402XXXXXXXX9016 IBER LIMOSTAR-FUENLABRADA,-11.50,ma
14,2024-07-15,COMPRA TARJ. 5402XXXXXXXX9016 IBER LIMOSTAR-FUENLABRADA,-35.95,ma
19,2024-07-04,COMPRA TARJ. 5402XXXXXXXX9016 CLINICA VETERINARIA ESCLA-LAS ROZAS DE,-7.00,ma
20,2024-07-01,COMPRA TARJ. 5402XXXXXXXX9016 MARKET LAS ROZA-LAS ROZAS,-7.00,ma


In [303]:
print()
display(Markdown("# Mine excluded from calculation"))
display(filter_statements_by_date(my_excluded, month, year))

print()
display(Markdown("# Mine indcluded calculation"))
display(filter_statements_by_date(my_included, month, year))

# Mine excluded from calculation

,date,name,value
1,2024-07-27,Reintegro Contra Cuenta En Atm 004951820001 El 27/07/2024 A Las 13:50. .pan: 5489010369724776.,-20.00
2,2024-07-29,"Transferencia De Educadvisor, S.l., Concepto Abono Nomina 07/2024.",2105.89
3,2024-07-21,"Compra Mgp*wallapop S 0402593, Barcelona, Tarjeta 5489010369724776 , Comision 0,00",-20.80
4,2024-07-21,"Compra Aliexpress, London, Tarjeta 5489010369724776 , Comision 0,00",-48.25
6,2024-07-20,"Transaccion Contactless En Leroy Merlin La, Las Rozas, Tarj. :*724776",-25.98
7,2024-07-20,"Transaccion Contactless En Leroy Merlin La, Las Rozas, Tarj. :*724776",-18.73
9,2024-07-15,"Transaccion Contactless En El Corte Ingles, Madrid, Tarj. :*724776",-18.09
11,2024-07-07,"Compra Mgp*wallapop S 0402593, Barcelona, Tarjeta 5489010369724776 , Comision 0,00",-100.69
12,2024-07-07,"Transaccion Contactless En Metro De Madrid, Madrid, Tarj. :*724776",-6.10
13,2024-07-06,Reintegro Contra Cuenta En Atm 004951820020 El 06/07/2024 A Las 19:45. .pan: 5489010369724776.,-260.00


# Mine indcluded calculation

,date,name,value
0,2024-07-29,"Transaccion Contactless En Market Las Roza, Las Rozas, Tarj. :*724776",-3.73
5,2024-07-20,"Transaccion Contactless En Lidl Las Rozas, Rozas Las, Tarj. :*724776",-5.78
8,2024-07-17,"Transaccion Contactless En Market Las Roza, Las Rozas, Tarj. :*724776",-7.88
10,2024-07-15,"Transaccion Contactless En Alcampo28029-va, Madrid, Tarj. :*724776",-15.67
14,2024-07-05,"Transaccion Contactless En Market Las Roza, Las Rozas, Tarj. :*724776",-6.82
